![](https://media.licdn.com/dms/image/v2/D4E0BAQGumdWW8R2giQ/company-logo_200_200/B4EZ87y9pLK4AE-/0/1783414643504/health_data_research_service_logo?e=2147483647&v=beta&t=GxsLjCFo9AmdCTHkDLK622NOIF3hplTcipdZBeJbzPY)

# Technology Feasibility - Demo 1

## Objective of the demo

Demonstrate that we can access synthetic data held by different 'organisations' in different clouds through a single interface and query engine

---

## Scene Setting

Oxford University Hospitals, University Hospitals Sussex and University Hospital Birmingham are collaborating in order to analyse medication usage across a rare cancer, in this case penile cancer.

---

#### Demo

1. Look at the available data
2. Show where the data is stored
3. Answer a research question
4. Upcoming demos



#### Look at the available data


In [ ]:
from pyspark.sql import DataFrame
import pandas as pd

In [ ]:
## Import libraries and connect to the query engine

from pyspark.sql import SparkSession
from pyspark.sql import functions as f
spark = SparkSession.builder.appName("test").remote("sc://spark-connector-0-driver-svc:15002").getOrCreate()


In [ ]:
spark.sql("show tables in aws.acceptancetests").filter(f.col("tableName").startswith("ouh")).select(f.col("tableName")).show()
spark.sql("show tables in azure.acceptancetests").filter(f.col("tableName").startswith("uhs")).select(f.col("tableName")).show()
spark.sql("show tables in gcp.acceptancetests").filter(f.col("tableName").startswith("uhb")).select(f.col("tableName")).show()

### Show where the data is stored

In [ ]:
spark.sql("describe table extended aws.acceptancetests.ouh_conditions").filter("col_name = 'Location'").show(truncate=False)
spark.sql("describe table extended azure.acceptancetests.uhs_conditions").filter("col_name = 'Location'").show(truncate=False)
spark.sql("describe table extended gcp.acceptancetests.uhb_conditions").filter("col_name = 'Location'").show(truncate=False)



### Research Question

What combination of medications are patients diagnosed with stage 4 penile cancer prescribed?


In [ ]:
spark.sql(""" describe table gcp.acceptancetests.uhb_conditions""").show()
spark.sql(""" describe table gcp.acceptancetests.uhb_medications""").show()


In [ ]:
spark.sql("""
select count(distinct patient_id) as uhb_penile_cancer_patients from gcp.acceptancetests.uhb_conditions 
where snomed_code = "363515000" and stage = "Stage 4" """).show()

spark.sql("""
select count(distinct patient_id) as ouh_penile_cancer_patients from aws.acceptancetests.ouh_conditions 
where snomed_code = "363515000" and stage = "Stage 4" """).show()

spark.sql("""
select count(distinct patient_id) as uhs_penile_cancer_patients from azure.acceptancetests.uhs_conditions 
where snomed_code = "363515000" and stage = "Stage 4" """).show()

In [ ]:
def sql_generator(cloud,table1,table2):
    sql = f"""select B.patient_id,B.medication from (
select patient_id from {cloud}.acceptancetests.{table1} 
where snomed_code = "363515000" and stage = "Stage 4") A
left join 
(select patient_id,medication from {cloud}.acceptancetests.{table2}) B
on A.patient_id = B.patient_id
"""
    return sql

unioned_data = sql_generator('azure','uhs_conditions','uhs_medication') + " UNION All \n" + \
sql_generator('gcp','uhb_conditions','uhb_medications') + " UNION All \n" + \
sql_generator('aws','ouh_conditions','ouh_medications') 

all = spark.sql(unioned_data).toPandas()


In [ ]:

all.groupby('patient_id').agg(lambda x : "--".join(set(x))).groupby('medication').size()
        